# Defence Demo
## Selected Attack Mitigation Procedure

This notebook demonstrates the **input-transformation defence pipeline** against the PGD adversarial attack.

Defences implemented in `defend.py`:
- `jpeg` — JPEG re-compression (destroys high-frequency adversarial noise)
- `blur` — Gaussian blur (5×5 kernel)
- `squeeze` — Feature squeezing (4-bit colour depth reduction)
- `median` — Median filter (3×3)

> **Screenshot this notebook** for the *Mitigation Procedure* and *Output After Security Measures* sections.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from defend import jpeg_compression, gaussian_blur, feature_squeezing, median_filter, apply_defences
from attack import postprocess_count

print('Defence module loaded.')

In [ ]:
model = YOLO('../yolov8n.pt')

clean_img   = cv2.imread('../outputs/crowd.jpg')
attacked_img = cv2.imread('../outputs/crowd_attacked.jpg')

clean_count   = postprocess_count(model, clean_img)
attacked_count = postprocess_count(model, attacked_img)

print(f'Clean image count  : {clean_count}')
print(f'Attacked image count: {attacked_count}')

## 1. Apply Individual Defences

In [ ]:
defence_fns = {
    'JPEG (q=75)': jpeg_compression,
    'Gaussian Blur (5×5)': gaussian_blur,
    'Feature Squeezing (4-bit)': feature_squeezing,
    'Median Filter (3×3)': median_filter,
}

results = {}
for name, fn in defence_fns.items():
    defended = fn(attacked_img)
    count = postprocess_count(model, defended)
    results[name] = count
    print(f'{name:<30}: {count} people detected')

## 2. Apply Combined Defence (JPEG + Blur)

In [ ]:
defended_img = apply_defences(attacked_img, ['jpeg', 'blur'])
defended_count = postprocess_count(model, defended_img)

cv2.imwrite('../outputs/crowd_defended.jpg', defended_img)
print(f'Defended (JPEG + Blur) count: {defended_count}')
print(f'Recovery: +{defended_count - attacked_count} detections restored')

## 3. Visualise: Clean / Attacked / Defended

In [ ]:
def annotate(model, bgr):
    r = model(bgr, conf=0.5, classes=[0], verbose=False)[0]
    return cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB), len(r.boxes)

clean_ann, cc = annotate(model, clean_img)
atk_ann, ac   = annotate(model, attacked_img)
def_ann, dc   = annotate(model, defended_img)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(clean_ann);   axes[0].set_title(f'Clean — {cc} detected', fontsize=12); axes[0].axis('off')
axes[1].imshow(atk_ann);     axes[1].set_title(f'PGD Attack — {ac} detected', fontsize=12); axes[1].axis('off')
axes[2].imshow(def_ann);     axes[2].set_title(f'After Defence (JPEG+Blur) — {dc} detected', fontsize=12); axes[2].axis('off')
plt.suptitle('Clean → Attack → Defence Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/report_05_defense_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/report_05_defense_comparison.png')

## 4. Summary Table

In [ ]:
print('=' * 50)
print(f'{"Stage":<35} {"Count":>10}')
print('-' * 50)
print(f'{"Clean image (baseline)":<35} {clean_count:>10}')
print(f'{"After PGD attack (ε=0.08)":<35} {attacked_count:>10}')
print('-' * 50)
for name, cnt in results.items():
    print(f'  Defence: {name:<25} {cnt:>10}')
print('-' * 50)
print(f'{"Combined JPEG+Blur defence":<35} {defended_count:>10}')
print('=' * 50)
print(f'\nFull recovery: {defended_count == clean_count}')